# Enclave Evaluation — Benchmark Owner

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@ai-safety.org` | Trusted execution environment (sealed memory) |
| **Model Owner** | `model_owner@ai-safety.org` | Owns the private language-model weights (Qwen) |
| **Benchmark Owner** | (this notebook) | Owns the private Vietnamese-harm benchmark |
| **Researcher** | `researcher@ai-safety.org` | Writes the eval code, submits the job |

This notebook drives only the **Benchmark Owner** steps; the Model Owner and Researcher run their own notebooks in parallel.
Self-contained: the benchmark CSV is fetched from the public repo.

In [ ]:
!uv pip install -Uq "git+https://github.com/OpenMined/PySyft.git@dev#subdirectory=packages/syft-enclave"

## Setup

In [ ]:
import csv
import os
from pathlib import Path

import requests

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do

# This notebook is self-contained — no local checkout of blindfold needed. Code + benchmark
# are fetched from the public repo; weights come from Hugging Face. If the repo moves to the
# OpenMined org, change GITHUB_REPO. Pin GITHUB_REF to a commit/tag for a reproducible demo.
GITHUB_REPO = "khoaguin/blindfold"
GITHUB_REF = "main"
RAW = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}"

# Downloads land here — next to the notebook (in Colab: /content/...) so the fetched code +
# benchmark show up in the Files pane and can be opened live during the demo.
BASE = Path.cwd()


def fetch(repo_path: str, dest: Path) -> Path:
    # Download one file from the public repo into `dest`.
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"{RAW}/{repo_path}"
    resp = requests.get(url, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"could not fetch {url} (HTTP {resp.status_code}) — is {GITHUB_REPO} public yet?")
    dest.write_bytes(resp.content)
    return dest

In [ ]:
ENCLAVE_EMAIL         = "enclave@ai-safety.org"
RESEARCHER_EMAIL      = "researcher@ai-safety.org"

print(f"  Enclave: {ENCLAVE_EMAIL}  |  Researcher: {RESEARCHER_EMAIL}")

## Step 0 — Log in as Benchmark Owner

In [ ]:
benchmark_owner = login_do()
print(f"  Benchmark Owner : {benchmark_owner.email}")

In [ ]:
# Optionally clear state for a fresh run
benchmark_owner._manager.delete_syftbox()
benchmark_owner._manager.peer_manager.write_own_version()

### Launch the enclave

Start the enclave service for `enclave@ai-safety.org` out-of-band. It runs the job
automatically once **both** owners approve — no notebook drives it.

## Step 1 — Peer with the Enclave

In [ ]:
benchmark_owner.add_peer(ENCLAVE_EMAIL)
benchmark_owner.sync()
print(f"  Benchmark Owner peered with enclave ({ENCLAVE_EMAIL})")

### Step 1.1 — Wait for the Researcher peer request, then approve

The Researcher notebook adds you as a peer. Re-run the cell below until their request appears, then approve.

In [ ]:
benchmark_owner.sync()
benchmark_owner.peers

In [ ]:
benchmark_owner.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)
benchmark_owner.sync()
print("  Researcher peer approved")

### Step 1.2 — Attest enclave's identity

In [ ]:
# Wait for the enclave to accept the peer request, then attest
benchmark_owner.attest_peer(ENCLAVE_EMAIL)

## Step 2.1 — Benchmark assets

The full bilingual EN+VN benchmark is fetched into **`./benchmark/benchmark.csv`** (visible in the
Files pane). The mock is the first 3 rows (`./benchmark/benchmark_mock.csv`) — what the researcher
develops against; the full file is the private hidden set.

In [ ]:
# Full benchmark → ./benchmark/benchmark.csv (visible).
BENCH_DIR = BASE / "benchmark"
bench_csv = fetch("data/benchmark.csv", BENCH_DIR / "benchmark.csv")
ALL_ROWS = list(csv.DictReader(open(bench_csv, encoding="utf-8")))
FIELDS = list(ALL_ROWS[0].keys())


def write_csv(rows: list[dict], name: str) -> Path:
    p = BENCH_DIR / name
    with open(p, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows(rows)
    return p


print(f"  benchmark -> {BENCH_DIR}/  ({len(ALL_ROWS)} rows)")

## Step 2.2 — Upload the evaluation benchmark

* **mock**: 3 sample rows — the researcher inspects these to develop their eval
* **private**: the full bilingual EN+VN benchmark — only the enclave sees the full hidden set

In [ ]:
benchmark_owner.create_dataset(
    name="vn_harm_benchmark",
    mock_path=write_csv(ALL_ROWS[:3], "benchmark_mock.csv"),
    private_path=bench_csv,
    summary="Vietnamese-harm refusal benchmark (EN+VN): scam, medical, jailbreak, benign controls",
    users=[RESEARCHER_EMAIL, ENCLAVE_EMAIL],
    upload_private=True,
    sync=False,
)
benchmark_owner.share_private_dataset("vn_harm_benchmark", ENCLAVE_EMAIL)
benchmark_owner.sync()
print(f"  Benchmark Owner uploaded 'vn_harm_benchmark'  (mock: 3 rows, private: {len(ALL_ROWS)} rows)")

## Step 3 — Wait for the Researcher to submit the job, then approve

The Researcher submits `vn_harm_audit` to the enclave. Re-sync until it appears, inspect the
submitted code, then approve. The enclave only runs once **both** owners approve.

In [ ]:
JOB_NAME = "vn_harm_audit"
benchmark_owner.sync()
benchmark_owner_job = benchmark_owner.jobs[JOB_NAME]
print(f"  Benchmark Owner sees '{JOB_NAME}'  status={benchmark_owner_job.status}")
benchmark_owner_job  # inspect the submitted code before approving

In [ ]:
benchmark_owner.approve_job(benchmark_owner_job)
benchmark_owner.sync()
print("  Benchmark Owner approved")

## Step 4 — Receive results

Results arrive here because the Researcher submitted with `share_results_with_do=True`.

In [ ]:
benchmark_owner.sync()
benchmark_owner_job = benchmark_owner.jobs[JOB_NAME]
print(f"  Output files : {benchmark_owner_job.output_paths}")
assert len(benchmark_owner_job.output_paths) > 0